In [1]:
import os
import glob
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from extract_features import extract_custom_features

In [2]:
# ==========================================
# 1. LOAD DATA
# ==========================================
DATA_DIR = "./Data"
IMAGE_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

def list_images(folder):
    paths = []
    for ext in IMAGE_EXTS:
        paths.extend(glob.glob(os.path.join(DATA_DIR, folder, ext)))
        # also catch uppercase extensions
        paths.extend(glob.glob(os.path.join(DATA_DIR, folder, ext.upper())))
    return sorted(set(paths))

real_paths = list_images("real_art")
ai_paths = list_images("ai_images")

image_paths = real_paths + ai_paths
labels = [0] * len(real_paths) + [1] * len(ai_paths)  # 0 for real, 1 for AI

print(f"Tổng số ảnh: {len(image_paths)} (Real: {len(real_paths)}, AI: {len(ai_paths)})")

Tổng số ảnh: 155015 (Real: 50000, AI: 105015)


In [3]:
# ==========================================
# 2. EXTRACT FEATURES
# ==========================================
X_features = []
y_labels = []
saved_paths = []  # track which paths were actually processed

print("Bắt đầu trích xuất đặc trưng...")
for i in tqdm(range(len(image_paths)), desc="Processing Images"):
    img_path = image_paths[i]
    image = cv2.imread(img_path)

    if image is None:
        print(f"Lỗi đọc ảnh: {img_path}. Bỏ qua.")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (256, 256))

    features = extract_custom_features(image)

    if features is not None:
        X_features.append(features)
        y_labels.append(labels[i])
        saved_paths.append(img_path)

X_features = np.array(X_features, dtype=np.float32)
y_labels = np.array(y_labels)

print(f"\nĐã trích xuất xong! Kích thước ma trận dữ liệu (Số ảnh, Số đặc trưng): {X_features.shape}")

Bắt đầu trích xuất đặc trưng...


Processing Images: 100%|██████████| 155015/155015 [49:30<00:00, 52.19it/s]



Đã trích xuất xong! Kích thước ma trận dữ liệu (Số ảnh, Số đặc trưng): (155015, 51)


In [4]:
# ==========================================
# 2b. SAVE EXTRACTED FEATURES TO CSV
# ==========================================
import csv as _csv

FEATURES_CSV = "featured_extracted.csv"

n_features = X_features.shape[1]
header = ["filepath", "label"] + [f"f{i}" for i in range(n_features)]

with open(FEATURES_CSV, "w", newline="", encoding="utf-8") as f:
    writer = _csv.writer(f)
    writer.writerow(header)
    for path, label, feats in zip(saved_paths, y_labels, X_features):
        writer.writerow([path, int(label), *feats.tolist()])

print(f"Saved {len(saved_paths)} rows ({n_features} features each) to {FEATURES_CSV}")

Saved 155015 rows (51 features each) to featured_extracted.csv


In [5]:
# ==========================================
# 3. DATA SPLIT AND NORMALIZATION
# ==========================================
USING_MODEL = "RF"  # "SVM"

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# ==========================================
# 4. TRAINING
# ==========================================

print("Bắt đầu huấn luyện mô hình...")

if USING_MODEL == "RF":
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) 
else:
    model = SVC(kernel='rbf', C=1.0, random_state=42)

model.fit(X_train_scaled, y_train)
print("Huấn luyện xong!\n")

# 5. Đánh giá trên tập Test
print("--- KẾT QUẢ ĐÁNH GIÁ (TEST SET) ---")
y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Báo cáo phân loại chi tiết:")
print(classification_report(y_test, y_pred, target_names=["Real Art (0)", "AI Art (1)"]))

# (Tùy chọn) In thêm Ma trận nhầm lẫn để xem máy hay đoán nhầm Real thành AI hay ngược lại
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Bắt đầu huấn luyện mô hình...
Huấn luyện xong!

--- KẾT QUẢ ĐÁNH GIÁ (TEST SET) ---
Accuracy: 97.41%

Báo cáo phân loại chi tiết:
              precision    recall  f1-score   support

Real Art (0)       0.97      0.95      0.96     10000
  AI Art (1)       0.98      0.98      0.98     21003

    accuracy                           0.97     31003
   macro avg       0.97      0.97      0.97     31003
weighted avg       0.97      0.97      0.97     31003

Confusion Matrix:
[[ 9514   486]
 [  318 20685]]
